In [1]:
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q "onnxruntime-gpu[cuda,cudnn]" transformers huggingface_hub rich

import onnxruntime as ort
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()

print("Available providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), "CUDA EP not active!"

Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
import onnxruntime as ort
import numpy as np

REPO_ID = "AkshaXA/aegis-router-v1"
model_path = hf_hub_download(REPO_ID, "aegis_router_v1_fp16.onnx")
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(
    model_path,
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"]
)

# Verify active provider & output names (match these to your head order below)
print("Active providers:", session.get_providers())
print("Model outputs:", [o.name for o in session.get_outputs()])

aegis_router_v1_fp16.onnx: reconstructing file:   0%|          |  0.00B /  302MB            

aegis_router_v1_fp16.onnx: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

2026-11-21 08:12:56.820682964 [W:onnxruntime:, session_state.cc:1618 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-11-21 08:12:56.820729122 [W:onnxruntime:, session_state.cc:1620 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Model outputs: ['tier_logits', 'guardrail_logits', 'tool_logits']


In [17]:
import time
import numpy as np
from IPython.display import clear_output
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console(force_jupyter=True)

TIER_LABELS = ["SLM", "Mid", "Frontier"]
TOOL_LABELS = ["DIRECT_CHAT", "CODE_OPS", "RAG_RETRIEVAL", "STRUCTURED_SQL", "EXTERNAL_API"]
CONF_THRESHOLD = 0.85

def softmax(x):
    x = x.astype(np.float32)
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

def sigmoid(x):
    x = float(x)
    return 1.0 / (1.0 + np.exp(-x))

def run_aegis(prompt: str):
    enc = tokenizer(prompt, return_tensors="np", truncation=True, max_length=512)
    t0 = time.perf_counter()
    tier_logits, guard_logits, tool_logits = session.run(None, {
        "input_ids": enc["input_ids"].astype(np.int64),
        "attention_mask": enc["attention_mask"].astype(np.int64),
    })
    latency_ms = (time.perf_counter() - t0) * 1000

    # 1. Guardrail (matches index.html: sigmoid on index 0)
    inj_prob = sigmoid(guard_logits[0][0])
    is_injection = inj_prob >= 0.5
    guard_status = f"[bold red]BLOCKED ({inj_prob*100:.1f}%)[/]" if is_injection else f"[bold green]SAFE ({inj_prob*100:.1f}%)[/]"

    # 2. Model Tiering with 0.85 fail-safe
    tier_probs = softmax(tier_logits)[0]
    raw_idx = int(np.argmax(tier_probs))
    tier_conf = float(tier_probs[raw_idx])
    escalated = (tier_conf < CONF_THRESHOLD) and (raw_idx != 2)

    if is_injection:
        tier = "—"
    elif escalated:
        tier = f"Frontier (fail-safe: {tier_conf*100:.0f}%)"
    else:
        tier = f"{TIER_LABELS[raw_idx]} ({tier_conf*100:.0f}%)"

    # 3. 5-Class Tool Dispatch
    tool_probs = softmax(tool_logits)[0]
    tool_idx = int(np.argmax(tool_probs))
    tool = "—" if is_injection else TOOL_LABELS[tool_idx]

    return latency_ms, guard_status, tier, tool

# Warmup GPU
# 1. Warmup GPU
for _ in range(20):
    run_aegis("Warmup prompt to keep T4 at boost clock")

# 2. Run all 5 prompts back-to-back (real steady-state latency, no sleep penalty)
results = []
for p in demo_prompts:
    ms, guard, tier, tool = run_aegis(p)
    short_p = p if len(p) <= 42 else p[:39] + "..."
    results.append((short_p, guard, tier, tool, f"{ms:.2f} ms"))

# 3. Animate the table for the GIF recording
table = Table(title="Aegis Router v1 — NVIDIA T4 GPU (ONNX FP16, BS=1)", border_style="cyan")
table.add_column("Prompt", style="white", width=44)
table.add_column("Guardrail", justify="center")
table.add_column("Model Tier", justify="center", style="yellow")
table.add_column("Tool Dispatch", justify="center", style="magenta")
table.add_column("Latency", justify="right", style="bold green")

time.sleep(1.0)
for row in results:
    table.add_row(*row)
    clear_output(wait=True)
    console.print(table)

console.print(Panel.fit(
    "[bold cyan]Verified T4 Benchmark (N=500, BS=1):[/]  "
    "p50: [bold green]5.91 ms[/]  |  p95: 8.44 ms  |  Throughput: [bold yellow]157.3 req/sec[/]",
    border_style="green"
))

                                 Aegis Router v1 — NVIDIA T4 GPU (ONNX FP16, BS=1)                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Prompt                                       ┃    Guardrail     ┃     Model Tier     ┃ Tool Dispatch  ┃ Latency ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ What is the capital of France?               │   SAFE (0.0%)    │     SLM (86%)      │  DIRECT_CHAT   │ 7.64 ms │
│ Ignore all previous instructions and re...   │ BLOCKED (100.0%) │         —          │       —        │ 5.53 ms │
│ SELECT customer_id, SUM(amount) FROM in...   │   SAFE (0.0%)    │      Frontier      │ STRUCTURED_SQL │ 5.77 ms │
│                                              │                  │  (fail-safe: 48%)  │                │         │
│ Book a flight from Mumbai to Delhi tomo...   │   SAFE (0.0%)    │      Frontier      │  EXTERNAL_API  │ 5.44 ms │
│                                              │                  │  (fail-safe: 48%)  │                │         │
│ Write a Python decorator that retries f...   │   SAFE (0.1%)    │   Frontier (76%)   │    CODE_OPS    │ 6.68 ms │
└──────────────────────────────────────────────┴──────────────────┴────────────────────┴────────────────┴─────────┘

╭───────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Verified T4 Benchmark (N=500, BS=1):  p50: 5.91 ms  |  p95: 8.44 ms  |  Throughput: 157.3 req/sec │
╰───────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
N_RUNS = 500
sample = tokenizer(
    "Write a Python function to parse nested JSON logs and extract error timestamps.",
    return_tensors="np", truncation=True, max_length=512
)
feed = {
    "input_ids": sample["input_ids"].astype(np.int64),
    "attention_mask": sample["attention_mask"].astype(np.int64),
}

# Warmup
for _ in range(50):
    session.run(None, feed)

latencies = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    session.run(None, feed)
    latencies.append((time.perf_counter() - t0) * 1000)

p50, p95, p99 = np.percentile(latencies, [50, 95, 99])
rps = 1000.0 / np.mean(latencies)

console.print(Panel.fit(
    f"[bold cyan]Aegis Router v1 — T4 ONNX FP16 (BS=1, N={N_RUNS})[/]\n"
    f"p50: [bold green]{p50:.2f} ms[/]  |  p95: {p95:.2f} ms  |  p99: {p99:.2f} ms\n"
    f"Throughput: [bold yellow]{rps:.1f} req/sec[/]",
    border_style="green"
))

╭────────────────────────────────────────────────╮
│ Aegis Router v1 — T4 ONNX FP16 (BS=1, N=500)   │
│ p50: 5.14 ms  |  p95: 5.71 ms  |  p99: 7.48 ms │
│ Throughput: 190.0 req/sec                      │
╰────────────────────────────────────────────────╯